# 01 – Data Download, Parsing & Validation
**Project:** Robust Outlier Detection in High-Dimensional Sensor Data from IoT Networks

**Dataset:** UCI *Gas Sensor Array Drift Dataset at Different Concentrations* (id 270)
https://archive.ics.uci.edu/dataset/270/gas+sensor+array+drift+dataset+at+different+concentrations

**Important caveats (document these in the report)**
- This is a *processed multivariate gas-sensor dataset* (16 sensors × 8 derived features = 128 features), **not** raw temperature/humidity/pressure IoT time series.
- The gas class (1–6) is the **gas type, not an anomaly label**. Anomalies are injected later with known ground truth.
- The system detects unusual multivariate sensor patterns (faults, drift, unusual conditions); it does **not** directly detect gas leaks.

**Rule:** if validation fails, STOP and diagnose the file format. Do not continue to anomaly injection.

## 1. Setup
Runs locally from `notebooks/` or the project root. On Google Colab, upload `data_loader.py` when prompted.

In [ ]:
import sys
from pathlib import Path

# Locate src/data_loader.py (local) or fall back to an upload (Colab)
here = Path.cwd()
candidates = [here, here.parent]
root = next((p for p in candidates if (p / "src" / "data_loader.py").exists()), None)

if root is None:
    try:
        from google.colab import files  # only exists on Colab
        (here / "src").mkdir(exist_ok=True)
        (here / "src" / "__init__.py").touch()
        print("Upload src/data_loader.py")
        up = files.upload()
        (here / "src" / "data_loader.py").write_bytes(next(iter(up.values())))
        root = here
    except ImportError:
        raise FileNotFoundError("Could not find src/data_loader.py. Run from the project folder.")

sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
from src import data_loader as dl

pd.set_option("display.max_columns", 20)
print("Project root:", root)
print("Raw dir     :", dl.RAW_DIR)

## 2. Download the official UCI archive
Downloads the zip, extracts it (including nested zips) and locates `batch1.dat … batch10.dat`. If the automatic download fails, download manually from the UCI page and extract into `data/raw/`.

In [ ]:
files = dl.download_dataset(dl.RAW_DIR)
for b, p in files.items():
    print(f"batch{b:<2d} -> {p.relative_to(root)}  ({p.stat().st_size/1e6:.2f} MB)")

## 3. Inspect the raw format before parsing
The parser assumes one measurement per line: `class;concentration 1:v … 128:v` (129 tokens). This cell checks that assumption against the real files.

In [ ]:
dl.diagnose_files(dl.RAW_DIR)

## 4. Parse

In [ ]:
df = dl.load_dataframe(dl.RAW_DIR, validate=False)   # validate explicitly in the next cell
print("Parsed shape:", df.shape)

## 5. Hard validation
Expected: **13,910 rows × 132 columns** and exact batch counts 445, 1244, 1586, 161, 197, 2300, 3613, 294, 470, 3600. A mismatch raises `DatasetValidationError`, which halts the notebook.

In [ ]:
report = dl.validate_dataframe(df, dl.RAW_DIR)   # raises on any mismatch
dl.print_validation_summary(df, report)

### Batch counts: expected vs. parsed

In [ ]:
cmp = pd.DataFrame({
    "expected": pd.Series(dl.EXPECTED_BATCH_COUNTS),
    "parsed": df["Batch"].value_counts().sort_index(),
})
cmp["match"] = cmp["expected"] == cmp["parsed"]
cmp.loc["TOTAL"] = [cmp["expected"].sum(), cmp["parsed"].sum(), cmp["expected"].sum() == cmp["parsed"].sum()]
cmp

## 6. Schema

In [ ]:
print("Shape:", df.shape)
print("First 4 columns + 2 features:")
display(df.iloc[:5, :6])
print(df.dtypes.value_counts())

## 7. Feature naming (16 sensors × 8 derived features)
**Verify** the derived-feature order against the *Variables* table on the UCI page. If it differs, edit `DERIVED_FEATURE_NAMES` in `src/data_loader.py`.

In [ ]:
mapping = dl.feature_mapping_table()
display(mapping.head(16))
print("Total features:", len(mapping), "| unique names:", mapping.feature_name.nunique())

## 8. Sanity checks on the real data
Quick checks only; the full EDA is in notebook 03.

In [ ]:
print("Gas distribution:")
display(df["Gas"].value_counts().to_frame("count"))
print("Gas × Batch:")
display(pd.crosstab(df["Batch"], df["Gas"]))
print("Concentration summary (ppmv):")
display(df["Concentration"].describe().to_frame().T)
print("Missing feature values:", int(df[dl.build_feature_names()].isna().sum().sum()))
print("Exact duplicate rows  :", int(df.duplicated().sum()))

## 9. Save the validated dataset
The original data stays untouched; all experiments will use copies.

In [ ]:
dl.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(dl.PARSED_CSV, index=False)
mapping.to_csv(dl.PROCESSED_DIR / "feature_mapping.csv", index=False)
print("Saved:", dl.PARSED_CSV)
print("Saved:", dl.PROCESSED_DIR / "feature_mapping.csv")

## Result log (fill in after running)
| Check | Result |
|---|---|
| Shape | *not yet executed* |
| Batch counts match | *not yet executed* |
| Missing values | *not yet executed* |
| Duplicates | *not yet executed* |

**Next:** `03_eda.ipynb` (EDA + pre-injection outlier visualisation).